In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from matplotlib.lines import Line2D
import sys

In [ ]:

from pyprojroot import here

import importlib.util

_spec = importlib.util.spec_from_file_location("opinion_functions", here() / "src" / "opinion_functions.py")
_mod = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_mod)
fun = _mod

_spec = importlib.util.spec_from_file_location("generate_homophilic_graph_symmetric", here() / "src" / "generate_homophilic_graph_symmetric.py")
_mod = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_mod)
generate_homophilic_graph_symmetric = _mod



In [ ]:
colors    = ['#19bdff', '#f2d138']
greycolor = '#626262'
myblack   = '#222222'

plt.rc('font', **{'family': 'sans-serif', 'sans-serif': ['Arial']})
plt.rcParams['font.family'] = 'Arial'

SMALL_SIZE  = 10
MEDIUM_SIZE = 10
BIGGER_SIZE = 10

plt.rc('font',   size=SMALL_SIZE)
plt.rc('axes',   titlesize=MEDIUM_SIZE)
plt.rc('axes',   labelsize=MEDIUM_SIZE)
plt.rc('xtick',  labelsize=SMALL_SIZE)
plt.rc('ytick',  labelsize=SMALL_SIZE)
plt.rc('legend', fontsize=SMALL_SIZE)
plt.rc('figure', titlesize=BIGGER_SIZE)

def customize_plot(axis, color='black', title_font_size=10, axis_font_size=10):
    axis.spines['bottom'].set_color(color)
    axis.spines['left'].set_color(color)
    axis.tick_params(axis='x', colors=color, labelsize=axis_font_size)
    axis.tick_params(axis='y', colors=color, labelsize=axis_font_size)
    axis.title.set_fontsize(title_font_size)
    sns.despine()

In [ ]:
# Define hvec
hvec = [0, 0.1, 0.5, 0.75, 0.8, 0.9, 1]

majority_frac    = 0.66
majority_percent = majority_frac * 100   # 66

num_agents = 1000

# Empirical misperception range: true mean 66%, survey estimates 37–43%
# → beta_overall = 66 - 43 = 23 to 66 - 37 = 29
BETA_EMPIRICAL_LO = 23
BETA_EMPIRICAL_HI = 29

dfs = {}

def empirical_xspan(x_pct, beta_overall):
    """Return (x_entry, x_exit) where beta_overall ∈ [LO, HI], or None."""
    inside = (beta_overall >= BETA_EMPIRICAL_LO) & (beta_overall <= BETA_EMPIRICAL_HI)
    if not inside.any():
        return None
    x_entry = x_pct[np.argmax(inside)]
    x_exit  = x_pct[len(inside) - 1 - np.argmax(inside[::-1])]
    return x_entry, x_exit

In [ ]:
# Loop through each value in hvec and load the corresponding CSV
for h in hvec:
    # Construct the file name based on h
    file_path = str(here() / '06_swap_simulations' / 'swapped_data' / f'swap_sim_h_{str(int(h*100))}.csv')
    
    # Check if the file exists and load it
    if os.path.exists(file_path):
        # Use the integer value of h * 100 as the key (e.g., df0, df10, etc.)
        dfs[f'df{int(h*100)}'] = pd.read_csv(file_path)
        print(f"Loaded {file_path} into df{int(h*100)}")
    else:
        print(f"File {file_path} does not exist.")

In [ ]:
dot_size = 10
df    = dfs['df75']
x_pct = df['swap_count'] * 100 / num_agents

beta_overall_pct = majority_percent - df['mean_opinion_percent']
beta_min_pct     = majority_percent - df['mean_minority_opinion_percent']
beta_maj_pct     = majority_percent - df['mean_majority_opinion_percent']

beta_overall = beta_overall_pct / 100
beta_min     = beta_min_pct     / 100
beta_maj     = beta_maj_pct     / 100

fig, axes = plt.subplots(1, 1, figsize=(3.42, 2.5), dpi=300)

axes.scatter(x_pct, beta_overall, color='grey',    s=dot_size)
axes.scatter(x_pct, beta_min,     color=colors[1], s=dot_size)
axes.scatter(x_pct, beta_maj,     color=colors[0], s=dot_size)

span = empirical_xspan(x_pct.values, beta_overall_pct.values)
if span is not None:
    axes.axvspan(span[0], span[1], color='#a83232', alpha=0.3, edgecolor=None)
    axes.text(span[0] + 0.2, -0.09, r"realistic $\beta$",
              rotation=90, verticalalignment='center', fontsize=8, color='#a83232')

axes.set_xlabel('Percent of nodes swapped')
axes.set_ylabel(r'misperception $\beta$')
axes.set_ylim(-0.33, 0.66)
axes.set_xticks([0, 10, 20, 30])
axes.set_title('A. Media bias and misperception')

customize_plot(axis=axes)

handles = [
    Line2D([0], [0], color='none', marker='s', markersize=7, label='overall',
           markerfacecolor=greycolor, markeredgewidth=0),
    Line2D([0], [0], color='none', marker='s', markersize=7, label='support',
           markerfacecolor=colors[0], markeredgewidth=0),
    Line2D([0], [0], color='none', marker='s', markersize=7, label='oppose',
           markerfacecolor=colors[1], markeredgewidth=0),
]
axes.legend(handles=handles, loc='lower right', frameon=False, fontsize=SMALL_SIZE)

plt.tight_layout()
plt.savefig(str(here() / "figures" / "single_panel_high_homophily_without_legend.pdf"),
            format='pdf', bbox_inches='tight')
plt.show()
